## 1. Environment Setup

In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes
!pip install -q scikit-learn matplotlib seaborn

In [ ]:
import os
import sys
import torch
import numpy as np
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from utils.data_loader import (
    load_defect_dataset,
    format_dataset_for_model_id,
    build_prompt_for_model,
    build_result_model_name,
    build_results_dir,
    summarize_label_distribution,
    build_error_records,
    split_error_types,
)
from utils.evaluation import (
    evaluate_predictions,
    parse_prediction_with_fallback,
    build_evaluation_payload,
    compare_saved_results,
    print_result_comparison,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


## 2. Configuration

In [ ]:
MODEL_ID = "codellama/CodeLlama-7b-Instruct-hf"
OUTPUT_DIR = "./codellama-defect-finetuned"
RESULTS_DIR = build_results_dir(MODEL_ID)
MODEL_NAME = build_result_model_name(MODEL_ID)

MAX_SEQ_LENGTH = 1024
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01


## 3. Load and Prepare Dataset

In [ ]:
train_data, val_data, test_data = load_defect_dataset()

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# Label distribution
train_labels = [s["target"] for s in train_data]
print(f"\nTrain label distribution: 0={train_labels.count(0)}, 1={train_labels.count(1)}")

In [ ]:
train_formatted = format_dataset_for_model_id(train_data, MODEL_ID)
val_formatted = format_dataset_for_model_id(val_data, MODEL_ID)

print("Train label distribution:", summarize_label_distribution(train_data))
print("Validation label distribution:", summarize_label_distribution(val_data))
print("Sample formatted prompt:")
print(train_formatted[0]["text"][:500])
print("...")


## 4. Load Model with 4-bit Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = prepare_model_for_kbit_training(model)
print(f"Model loaded. Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 5. Apply LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    dataset_text_field="text",
)

print("Starting training...")
trainer.train()

In [ ]:
trainer.save_model(os.path.join(OUTPUT_DIR, "best"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best"))
print(f"Best model saved to {OUTPUT_DIR}/best")

## 7. Evaluation on Test Set

In [ ]:
from tqdm import tqdm

model.eval()
device = next(model.parameters()).device
generation_config = {"max_new_tokens": 5, "do_sample": False, "temperature": 1.0}

y_true = []
y_pred = []
failed_parses = []

for i, sample in enumerate(tqdm(test_data, desc="Evaluating")):
    prompt = build_prompt_for_model(MODEL_ID, sample["func"], tokenizer=tokenizer, examples=None)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_config)

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred, parse_failed = parse_prediction_with_fallback(generated, sample["target"])

    if parse_failed:
        failed_parses.append({"index": i, "output": generated, "label": int(sample["target"])})

    y_true.append(int(sample["target"]))
    y_pred.append(pred)

print(f"\nFailed to parse: {len(failed_parses)} / {len(test_data)}")
if failed_parses[:5]:
    print("Sample failed outputs:")
    for fp in failed_parses[:5]:
        print(f"  idx={fp['index']}, output='{fp['output']}', true_label={fp['label']}")


In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

errors = build_error_records(test_data, y_true, y_pred, max_chars=200)
false_positives, false_negatives = split_error_types(errors)
payload = build_evaluation_payload(
    failed_parses=failed_parses,
    errors=errors,
    false_positives=false_positives,
    false_negatives=false_negatives,
)

metrics = evaluate_predictions(
    y_true, y_pred,
    model_name=MODEL_NAME,
    strategy="finetuning",
    save_dir=RESULTS_DIR,
    extra_data=payload,
)


## 8. Error Analysis

In [ ]:
print(f"Total errors: {len(errors)}")
print(f"False Positives (predicted defective but safe): {len(false_positives)}")
print(f"False Negatives (predicted safe but defective): {len(false_negatives)}")

print("\n--- Sample False Negatives (missed bugs) ---")
for e in false_negatives[:3]:
    print(f"\nIndex: {e['index']}")
    print(f"Code preview: {e['code']}...")

print("\n--- Sample False Positives (false alarms) ---")
for e in false_positives[:3]:
    print(f"\nIndex: {e['index']}")
    print(f"Code preview: {e['code']}...")


## 9. Compare Saved CodeLlama Results


In [ ]:
results = compare_saved_results(RESULTS_DIR, MODEL_NAME)
if results:
    print("\n" + "="*60)
    print("  ALL CODELLAMA RESULTS")
    print("="*60)
    print_result_comparison(results)
